# 模組 5：進階例外處理 — raise 與自定義例外

目標：學習主動拋出例外、例外鏈接，以及建立自定義例外類別。

---

## ⚾ 投手 vs 捕手

| 角色 | 語法 | 職責 |
|---|---|---|
| **投手（進攻方）** | `raise` | 程式狀態不對時，主動拋出錯誤 |
| **捕手（防守方）** | `try/except` | 捕捉並處理飛過來的錯誤 |

---
## 1. 基礎拋出 — 丟出內建例外

從「例外目錄」挑選適合的例外型態，在條件不合理時主動拋出。

In [ ]:
def calculate_bmi(weight, height):
    if not isinstance(weight, (int, float)) or not isinstance(height, (int, float)):
        raise TypeError("體重和身高必須是數字！")
    if height <= 0:
        raise ValueError("身高必須大於 0")
    if weight <= 0:
        raise ValueError("體重必須大於 0")
    return weight / ((height / 100) ** 2)

# 試試看不同的參數：
# calculate_bmi(70, 175)    → 正常
# calculate_bmi(70, -5)     → ValueError
# calculate_bmi("重", 175)   → TypeError

try:
    result = calculate_bmi(70, -5)
    print(f"BMI = {result:.2f}")
except (TypeError, ValueError) as e:
    print(f"[輸入錯誤] {e}")

---
## 2. 無參數拋出 (Bare Raise) — 攔截後原封不動往上丟

攔截錯誤只是為了記錄 Log，處理完後仍讓上層知道出事了。

In [ ]:
def read_config(filepath):
    try:
        with open(filepath, "r") as f:
            return f.read()
    except FileNotFoundError:
        print("【系統日誌】警告！找不到設定檔。")  # 記錄 Log
        raise   # 不帶參數 → 原封不動再拋出 FileNotFoundError

try:
    read_config("config.json")
except FileNotFoundError as e:
    print(f"[上層捕捉到] {e}")

---
## 3. 例外鏈接 (Exception Chaining) — `raise ... from ...`

把底層技術錯誤「包裝」成商業邏輯錯誤，同時保留原始錯誤軌跡。

In [ ]:
class UserNotFoundError(Exception):
    pass

def get_user_data(user_id):
    database = {"001": "Alice", "002": "Bob"}
    try:
        return database[user_id]
    except KeyError as raw_error:
        # 技術層：KeyError → 包裝成商業層：UserNotFoundError
        raise UserNotFoundError(f"找不到 ID 為 {user_id} 的使用者") from raw_error

try:
    get_user_data("999")
except UserNotFoundError as e:
    print(f"[UserNotFoundError] {e}")
    print(f"原始原因：{e.__cause__}")  # 查看原始的 KeyError

---
## 4. 自定義例外類別 — 擴充「例外目錄」

In [ ]:
class BankError(Exception):
    """銀行系統所有錯誤的基礎類別"""
    pass

class InsufficientFundsError(BankError):
    """餘額不足"""
    pass

class InvalidAccountError(BankError):
    """帳號格式錯誤"""
    pass

# 確認繼承關係
print(InsufficientFundsError.__mro__)
# → InsufficientFundsError → BankError → Exception → BaseException → object

---
## 5. 🏦 終極整合：銀行轉帳系統

| 修改參數 | 觸發例外 | 被哪個 except 接住 |
|---|---|---|
| `amount="五十"` | `TypeError` | `except TypeError` |
| `to_account="123"` | `InvalidAccountError` | `except BankError`（父類別） |
| `amount=5000` | `InsufficientFundsError` | `except InsufficientFundsError` |
| `amount=500` | 無例外 | `else` 區塊執行 |

In [ ]:
class BankError(Exception):
    pass

class InsufficientFundsError(BankError):
    pass

class InvalidAccountError(BankError):
    pass

def transfer_money(from_account, to_account, amount, balance):
    if not isinstance(amount, (int, float)):
        raise TypeError("轉帳金額必須是數字！")
    if len(to_account) != 5:
        raise InvalidAccountError(f"帳號 {to_account} 格式錯誤，長度需為 5 碼。")
    if amount > balance:
        raise InsufficientFundsError(f"轉帳 {amount} 元失敗，餘額僅 {balance} 元。")
    return f"成功轉帳 {amount} 元至 {to_account}，剩餘 {balance - amount} 元。"

def run_atm_system():
    my_balance = 1000
    try:
        result = transfer_money("A0001", "B0002", 5000, my_balance)  # ← 修改這行
    except TypeError as e:
        print("🔴 【輸入格式錯誤】", e)
    except InsufficientFundsError as e:
        print("🟡 【交易拒絕】", e)
        print("   👉 請先存入足夠金額。")
    except BankError as e:
        print("🟠 【銀行業務異常】", e)
    except Exception as e:
        print("💥 【未知嚴重錯誤】", e)
        raise
    else:
        print("🟢", result)
    finally:
        print("--- 交易流程結束 ---")

run_atm_system()

---
## 6. 自定義例外帶附加屬性

例外類別可以攜帶更多資訊（如：是哪個欄位出錯），讓捕捉端能做更精準的處理。

In [ ]:
class ValidationError(Exception):
    def __init__(self, field, message):
        self.field = field                          # 哪個欄位出錯
        super().__init__(f"[{field}] {message}")    # 傳給父類別顯示

def validate_user(name, age):
    if not name:
        raise ValidationError(field="name", message="姓名不能為空")
    if not isinstance(age, int) or age < 0:
        raise ValidationError(field="age", message="年齡必須是正整數")

try:
    validate_user("", 25)
except ValidationError as e:
    print(f"驗證失敗：{e}")
    print(f"出錯欄位：{e.field}")   # 透過屬性取得欄位名稱，可以用來在 UI 上標紅色

---
## 7. `raise from None` — 切斷例外鏈

有時候你不想讓使用者看到底層技術細節，只顯示乾淨的錯誤訊息。

In [ ]:
class ConfigError(Exception):
    pass

config = {"host": "localhost"}   # 故意少了 port

# 比較兩種寫法
print("=== 保留例外鏈（raise ... from error）===")
try:
    try:
        port = config["port"]
    except KeyError as e:
        raise ConfigError("設定檔缺少 port") from e   # 保留原始 KeyError
except ConfigError as e:
    print(f"錯誤：{e}")
    print(f"原因：{e.__cause__}")    # → 'port'

print()
print("=== 切斷例外鏈（raise ... from None）===")
try:
    try:
        port = config["port"]
    except KeyError:
        raise ConfigError("設定檔缺少 port") from None  # 隱藏底層 KeyError
except ConfigError as e:
    print(f"錯誤：{e}")
    print(f"原因：{e.__cause__}")    # → None

---
## 8. 表單多欄位驗證 — 收集所有錯誤後統一拋出

一般驗證遇到第一個錯誤就停，這個模式是先把所有問題都找出來，最後一次告知使用者。

In [ ]:
class FormValidationError(Exception):
    """表單驗證錯誤，附帶所有欄位的錯誤清單"""
    def __init__(self, errors: list):
        self.errors = errors
        super().__init__(f"表單驗證失敗，共 {len(errors)} 個錯誤")

def validate_form(name, email, age):
    errors = []   # 先收集所有錯誤

    if not name:
        errors.append("姓名不能為空")
    if "@" not in email:
        errors.append("email 格式不正確")
    if not isinstance(age, int) or age < 18:
        errors.append("年齡必須是 18 歲以上的整數")

    if errors:
        raise FormValidationError(errors)   # 全部收完再一次拋出

try:
    validate_form("", "not-an-email", 15)   # 三個欄位全部錯
except FormValidationError as e:
    print(f"❌ {e}")
    for i, err in enumerate(e.errors, 1):
        print(f"   {i}. {err}")

---
## 9. 重試機制 (Retry Pattern)

連線失敗時自動重試，超過最大次數才真的拋出例外。

In [ ]:
import random

def unstable_connection():
    """模擬不穩定的連線：70% 機率失敗"""
    if random.random() < 0.7:
        raise ConnectionError("連線失敗")
    return "連線成功！"

def fetch_with_retry(max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            result = unstable_connection()
            print(f"第 {attempt} 次嘗試：{result}")
            return result
        except ConnectionError as e:
            print(f"第 {attempt} 次嘗試失敗：{e}")
            if attempt == max_retries:
                raise   # 最後一次失敗才真的拋出，讓上層知道徹底失敗了
            print(f"   → 重試中...")

try:
    fetch_with_retry(max_retries=3)
except ConnectionError:
    print("\n❌ 重試 3 次後仍無法連線，請稍後再試。")

---
## 10. 狀態機 (State Machine) — 非法狀態轉換

訂單有固定的流程，當發生非法的狀態跳轉時拋出例外。

In [ ]:
class InvalidStateError(Exception):
    def __init__(self, current, next_state):
        super().__init__(f"不能從 [{current}] 轉換到 [{next_state}]")

# 定義合法的狀態轉換路徑
VALID_TRANSITIONS = {
    "待付款": ["已付款", "已取消"],
    "已付款": ["已出貨"],
    "已出貨": ["已送達"],
    "已送達": [],           # 終態，不能再轉換
    "已取消": [],           # 終態
}

class Order:
    def __init__(self, order_id):
        self.order_id = order_id
        self.state = "待付款"   # 初始狀態

    def transition(self, next_state):
        allowed = VALID_TRANSITIONS.get(self.state, [])
        if next_state not in allowed:
            raise InvalidStateError(self.state, next_state)
        print(f"訂單 {self.order_id}：{self.state} → {next_state}")
        self.state = next_state

order = Order("ORD-001")

# 正常流程
order.transition("已付款")
order.transition("已出貨")

# 嘗試非法跳轉：已出貨 → 已取消（不允許）
try:
    order.transition("已取消")
except InvalidStateError as e:
    print(f"\n❌ {e}")

---
## 練習：自己設計一個例外系統

試著為「線上購物系統」設計例外類別並實作：

```
ShopError (基礎)
├── OutOfStockError    ← 商品缺貨
└── InvalidCouponError ← 無效優惠券
```

In [ ]:
# 在這裡實作你的購物系統例外
pass